In [ ]:
# conda activate anndata

import os
import scvi
import torch
import pandas as pd
import scanpy as sc
import anndata as ad

torch.set_num_threads(20)
# OMP_NUM_THREADS=20

pd.set_option("display.max_columns", None)

os.chdir("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/MOp")

Here I run scVI to get a latent representation of the single-cell data for visualization purposes

In [ ]:
adata = ad.read_h5ad("yao_2021_MOp_STAR_gene_counts.h5ad")

In [ ]:
adata.layers["counts"] = adata.X.copy()
adata.raw = adata  # Keep full dimension safe

In [ ]:
# Subset data to highly variable features

sc.pp.highly_variable_genes(
  adata,
  n_top_genes = 2000,
  subset = True,
  layer = "counts",
  flavor = "seurat_v3",
)

In [ ]:
# scVI model

scvi.model.SCVI.setup_anndata(adata, layer="counts")
model = scvi.model.SCVI(adata)
model.train()

In [ ]:
# Extract low dim rep.
latent = model.get_latent_representation()
adata.obsm["scVI"] = latent

In [ ]:
os.chdir("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/MOp")

# Save
model.save("data/yao_2021_MOp_STAR_model", overwrite=True)
adata.write("data/yao_2021_MOp_STAR_model/yao_2021_MOp_STAR_gene_counts_scVI.h5ad")